In [6]:
from pathlib import Path
import numpy as np
import geopandas as gpd
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
import contextily as cx
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
# Gevraagd wordt een dataframe voor de UNPAVED zoals dit:
#	    id	    total_area	lu_areas	surface_level	soiltype	surface_storage	infiltration_capacity	initial_gwd	meteo_area	px	py	boundary_node
# code												
# 15.0	15.0	1375	250 0 0 0 0 0 0 0 0 0 225 0 0 0 0 0	                16.93   107	10.000	100.000	1.20	15.0	199378	395163	lat_15.0
# 55.0	55.0	303875	124200 18000 0 0 0 0 0 0 0 150 68125 0 11875 0...	21.69	105	10.000	100.000	1.20	55.0	197488	392239	lat_55.0
# 56.0	56.0	13300	5400 0 0 0 0 0 0 0 0 0 4425 0 375 0 1150 0	        20.63	113	10.000	100.000	1.20	56.0	197789	392200	lat_56.0
# 57.0	57.0	60925	6550 725 0 22800 0 0 0 0 0 0 9375 0 4600 0 175 0	21.49	113	10.000	100.000	1.20	57.0	197982	392247	lat_57.0

# en een dataframe voor ernst zoals deze:
# 	    id	    cvo	            lv	        cvi	    cvs
# code					
# 15.0	15.0	300 2000 100000	0.0 1.0 2.0	300.00	5.00
# 55.0	55.0	300 2000 100000	0.0 1.0 2.0	300.00	5.00
# 56.0	56.0	300 2000 100000	0.0 1.0 2.0	300.00	5.00
# 57.0	57.0	300 2000 100000	0.0 1.0 2.0	300.00	5.00

In [9]:
# For land use and soil type a coding is prescribed. For landuse, the legend of the map is expected to be as follows: <br>
landuse_mapping = {
    "potatoes": 1,
    "wheat": 2,
    "sugar beet": 3,
    "corn": 4,
    "other crops": 5,
    "bulbous plants": 6,
    "orchard": 7,
    "grass": 8,
    "deciduous forest": 9,
    "coniferous forest": 10,
    "nature": 11,
    "barren": 12,
    "open water": 13,
    "built-up": 14,
    "greenhouses": 15
}

# For classes 1-12, the areas are calculated from the provided raster and remapped to the classification in the Sobek RR-tables.


# The coding for the soil types:<br>
soiltype_first_mapping = {
    "podzol": "Podzol (grof zand)"
}

soiltype_mapping = {
    "Veengrond met veraarde bovengrond": 1,
    "Veengrond met veraarde bovengrond, zand": 2,
    "Veengrond met kleidek": 3,
    "Veengrond met kleidek op zand": 4,
    "Veengrond met zanddek op zand": 5,
    "Veengrond op ongerijpte klei": 6,
    "Stuifzand": 7,
    "Podzol (Leemarm, fijn zand)": 8,
    "Podzol (zwak lemig, fijn zand)": 9,
    "Podzol (zwak lemig, fijn zand op grof zand)": 10,
    "Podzol (lemig keileem)": 11,
    "Enkeerd (zwak lemig, fijn zand)": 12,
    "Beekeerd (lemig fijn zand)": 13,
    "Podzol (grof zand)": 14,
    "Zavel": 15,
    "Lichte klei": 16,
    "Zware klei": 17,
    "Klei op veen": 18,
    "Klei op zand": 19,
    "Klei op grof zand": 20,
    "Leem": 21
}

# And surface elevation needs to be in m+NAP.

In [10]:
def generate_unpaved_df_from_rr_input(gdf):
    df_unpaved = pd.DataFrame()
    df_unpaved["code"] = gdf["ID_RR_KNOOP"]    #"unpaved_" + gdf["GFEIDENT"]
    df_unpaved["id"] = gdf["ID_RR_KNOOP"]      #"unpaved_" + gdf["GFEIDENT"]
    df_unpaved["total_area"] = gdf["Area_RR_unpaved_m2"].astype(int)
    df_unpaved["lu_areas"] = gdf["Area_RR_unpaved_m2"].astype(int).astype(str) + " 0"*15
    df_unpaved["surface_level"] = gdf["SurfaceLevel_mNAP"]
    df_unpaved["soiltype"] = 114              # soiltype_mapping + 100 # nog naar kijken of de code klopt
    df_unpaved["surface_storage"] = gdf["StorageOnLand_mm"]
    df_unpaved["infiltration_capacity"] = gdf["InfiltrationCapacity_mmph"]
    df_unpaved["initial_gwd"] = gdf["InitialGroundwaterLevel_mBelowSurface"]
    df_unpaved["meteo_area"] = gdf["MeteoStationName"]      #Kijken in koppeltabel
    df_unpaved["px"] = gdf.geometry.x
    df_unpaved["py"] = gdf.geometry.y
    df_unpaved["boundary_node"] = "lat_" + gdf["ID_RR_KNOOP"].astype(str)
    df_unpaved["boundary_waterlevel"] = gdf["OpenWaterLevelBoundary_mNAP"]
    df_unpaved = df_unpaved.set_index("code")
    return df_unpaved


def generate_ernst_df_from_rr_input(gdf, max_drainage_value=9999999):
    for col in ["FirstDrainResistance_d", "SecondDrainResistance_d", "ThirdDrainResistance_d", "OpenWaterHorizontalInflowResistance_d", "SurfaceOverlandFlowResistance_d"]:
        gdf.loc[gdf[col]>max_drainage_value, col] = max_drainage_value
    df_ernst = pd.DataFrame()
    df_ernst["code"] = gdf["ID_RR_KNOOP"]      # "ernst_" + gdf["GFEIDENT"]
    df_ernst["id"] = gdf["ID_RR_KNOOP"]        # "ernst_" + gdf["GFEIDENT"]
    df_ernst["cvo"] = gdf.apply(lambda row: " ".join([str(row["FirstDrainResistance_d"]), str(row["SecondDrainResistance_d"]), str(row["ThirdDrainResistance_d"])]), axis=1)
    gdf["FirstDrainLevel_m"] = gdf["SurfaceLevel_mNAP"] - gdf["FirstDrainLevel_mNAP"]
    gdf["SecondDrainLevel_m"] = gdf["SurfaceLevel_mNAP"] - gdf["SecondDrainLevel_mNAP"]
    gdf["ThirdDrainLevel_m"] = gdf["SurfaceLevel_mNAP"] - gdf["ThirdDrainLevel_mNAP"]
    df_ernst["lv"] = gdf.apply(lambda row: " ".join([str(row["FirstDrainLevel_m"]), str(row["SecondDrainLevel_m"]), str(row["ThirdDrainLevel_m"])]), axis=1)
    #df_ernst["cvi"] = 9999999999
    df_ernst["cvi"] = gdf["OpenWaterHorizontalInflowResistance_d"].astype(str) # 10 * een 9
    df_ernst["cvs"] = gdf["SurfaceOverlandFlowResistance_d"].astype(str)
    df_ernst = df_ernst.set_index("code")
    return df_ernst


def generate_rr_unpaved_ernst_from_input(dir_scenario_input, dir_scenario_output):
    if not Path(dir_scenario_output).exists():
        Path(dir_scenario_output).mkdir(parents=True)
    seasons = ["zomer", "winter"]
    gpkg_rr_input_zomer_winter = ["RR_input_ZOMER.gpkg", "RR_input_WINTER.gpkg"]

    gdf_rr_input = {}

    for season, gpkg_rr_input in zip(seasons, gpkg_rr_input_zomer_winter):
        print(f"- season: {season}")

        gdf = gpd.read_file(dir_scenario_input / gpkg_rr_input)
        gdf = gdf[~gdf["ID_RR_KNOOP"].duplicated()]

        df_ernst = generate_ernst_df_from_rr_input(gdf)
        df_unpaved = generate_unpaved_df_from_rr_input(gdf)
        gdf_unpaved = gpd.GeoDataFrame(df_unpaved, geometry=gpd.points_from_xy(df_unpaved.px, df_unpaved.py), crs=28992)

        gdf_rr_input[season] = {}
        gdf_rr_input[season]["input"] = gdf
        gdf_rr_input[season]["ernst"] = df_ernst
        gdf_rr_input[season]["unpaved"] = gdf_unpaved

        df_ernst.to_csv(dir_scenario_output / f"df_ernst_{season}.csv")
        df_unpaved.to_csv(dir_scenario_output / f"df_unpaved_{season}.csv")
        gdf_unpaved.to_file(dir_scenario_output / f"gdf_unpaved_{season}.gpkg", layer=f"gdf_unpaved_{season}", driver="GPKG")
    
    return gdf_rr_input   

In [11]:
from datetime import timedelta

def write_bui(df, outfile, timestep_seconds=3600):
    stations = list(df.columns)
    nstations = len(stations)

    start = df.index[0]
    end = df.index[-1]

    duration = end - start + timedelta(seconds=timestep_seconds)

    dd = duration.days
    hh, rem = divmod(duration.seconds, 3600)
    mm, ss = divmod(rem, 60)

    with open(outfile, "w") as f:
        # Header
        f.write(f"*Name of this file: {outfile}\n")
        f.write("*Date and time of construction: 00/00/2000 00:00:00.\n")
        f.write("1\n")
        f.write("*Aantal stations\n")
        f.write(f"{nstations}\n")
        f.write("*Namen van stations\n")

        for s in stations:
            f.write(f"'{s}'\n")

        f.write("*Aantal gebeurtenissen (omdat het 1 bui betreft is dit altijd 1)\n")
        f.write("*en het aantal seconden per waarnemingstijdstap\n")
        f.write(f"1 {timestep_seconds}\n")
        f.write("*Elke commentaarregel wordt begonnen met een * (asterisk).\n")
        f.write("*Eerste record bevat startdatum en -tijd, lengte van de gebeurtenis in dd hh mm ss\n")
        f.write("*Het format is: yyyymmdd:hhmmss:ddhhmmss\n")
        f.write("*Daarna voor elk station de neerslag in mm per tijdstap.\n")

        # Startrecord
        f.write(
            f"{start.year} {start.month} {start.day} "
            f"{start.hour} {start.minute} {start.second} "
            f"{dd} {hh} {mm} {ss}\n"
        )

        # Tijdstappen
        for _, row in df.iterrows():
            line = " ".join(f"{v:.3f}" for v in row.values)
            f.write(line + "\n")

#### BASIS DIRECTORIES

In [12]:
start_date = "2010-4-1"
end_date = "2018-12-31"
# end_date = "2018-4-11"
cut_period_month = 12

In [13]:
# INPUT vanuit WRIJ voor RR unpaved methode
dir_data = Path("..\\..\\WRIJ_RR_Unpaved_methode_01_data\\")
dir_input = Path("..\\..\\WRIJ_RR_Unpaved_methode_02_input")

In [14]:
dir_input_basis_data = dir_input / "basisdata"

project_areas_path = dir_input_basis_data / "gebieden.gpkg"
path_watergang = dir_input_basis_data / "watergang.gpkg"
path_afwateringseenheden = dir_input_basis_data / "afwateringseenheden.gpkg"

project_areas = gpd.read_file(project_areas_path, layer="gebieden")
watergang = gpd.read_file(path_watergang)
afwateringseenheden = gpd.read_file(path_afwateringseenheden)


In [15]:
#Functie om de geodataframe aan te maken voor de zomer en de wintersituaties en het vullen van de geodataframe
def prepare_rr_input(df, koppeltabel):
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["xcoor"], df["ycoor"]),
        crs="EPSG:28992"
    )

    gdf = gdf.merge(
        koppeltabel[["GFEIDENT", "txt_file"]],
        on="GFEIDENT",
        how="left"
    )

    gdf["MeteoStationName"] = gdf["txt_file"].str.replace(".txt", "", regex=False)
    gdf.drop(columns="txt_file", inplace=True)

    #gdf["FirstDrainResistance_d"] = 0
    #gdf["SecondDrainResistance_d"] = 0

    #gdf["FirstDrainLevel_mNAP"] = gdf["SurfaceLevel_mNAP"]
    #gdf["SecondDrainLevel_mNAP"] = gdf["SurfaceLevel_mNAP"]

    #gdf["OpenWaterHorizontalInflowResistance_d"] = 9999999999

    return gdf

#### GEBIEDEN

In [16]:
# AREAS
for i, area in project_areas.iterrows():
    dir_input_area = dir_input / "rr_input_area_scenario" / f"gebied_{area.area_id}"

    area_gdf = project_areas.iloc[[i]]
    area_gdf.to_file(dir_input_area / f"gebied.gpkg", layer=f"gebied", driver="GPKG")

#### PREPARE INPUT FOR COMPLETE AREA

In [17]:
# INPUT vanuit WRIJ voor RR unpaved methode
dir_data_scenarios = Path(dir_data, "rr_data_scenarios")
dir_input_scenarios = Path(dir_input, "rr_input_scenarios")

scenario_gdf_input = {}

list_scenarios = [p.name for p in list(Path(dir_data_scenarios, "scenarios").iterdir())]

for scenario in list_scenarios:
    print(f"Scenario: {scenario}")
    dir_scenario_input = dir_data_scenarios / "scenarios" / scenario
    dir_scenario_output = dir_input_scenarios / "scenarios" / scenario

    # inlezen input
    path_input_scen_zomer = dir_scenario_input / f"RRunpaved_KNOPEN_{scenario}_ZOMER.csv"
    path_input_scen_winter = dir_scenario_input / f"RRunpaved_KNOPEN_{scenario}_WINTER.csv"

    input_scen_zomer = pd.read_csv(path_input_scen_zomer, delimiter=';')
    input_scen_winter = pd.read_csv(path_input_scen_winter, delimiter=';')

    scen_zomer = gpd.GeoDataFrame(input_scen_zomer, geometry=gpd.points_from_xy(input_scen_zomer.xcoor, input_scen_zomer.ycoor, crs=28992))
    scen_winter = gpd.GeoDataFrame(input_scen_winter, geometry=gpd.points_from_xy(input_scen_winter.xcoor, input_scen_winter.ycoor, crs=28992))
    scen_zomer = scen_zomer.drop(columns=["GFEIDENT"]).sjoin(afwateringseenheden[["GFEIDENT", "geometry"]], how="left")
    scen_winter = scen_winter.drop(columns=["GFEIDENT"]).sjoin(afwateringseenheden[["GFEIDENT", "geometry"]], how="left")
    
    input_scen_zomer = scen_zomer.drop(columns="geometry")
    input_scen_winter = scen_winter.drop(columns="geometry")
    
    #inladen koppeltabel
    path_koppeltabel = Path(dir_data_scenarios, "meteo\\neerslag_tijdreeksen\\output_koppeltabel\\koppeltabel.csv")
    koppeltabel = pd.read_csv(path_koppeltabel, delimiter=",")
    
    # debug: check for duplicates in koppeltabel
    koppeltabel = koppeltabel.drop_duplicates(subset=["GFEIDENT"], keep="first")

    #wegschrijven RR input
    rr_input_zomer = prepare_rr_input(input_scen_zomer, koppeltabel)
    rr_input_winter = prepare_rr_input(input_scen_winter, koppeltabel)

    afwateringseenheden_laterals = afwateringseenheden.merge(rr_input_zomer[["GFEIDENT", "ID_RR_KNOOP"]], on="GFEIDENT", how="right")

    afwateringseenheden_laterals["code"] = afwateringseenheden_laterals["ID_RR_KNOOP"].astype(str)
    afwateringseenheden_laterals["globalid"] = afwateringseenheden_laterals["GLOBALID"].astype(str)
    afwateringseenheden_laterals["lateraleknoopid"] = "lat_" + afwateringseenheden_laterals["ID_RR_KNOOP"].astype(str)
    afwateringseenheden_laterals = afwateringseenheden_laterals[["code", "globalid", "lateraleknoopid", "geometry"]]
    afwateringseenheden_laterals = afwateringseenheden_laterals[~afwateringseenheden_laterals.geometry.isnull()]

    afwateringseenheden_laterals.to_file(dir_scenario_input / "afwateringseenheden_laterals.gpkg", driver="GPKG")

    list_rr_nodes_codes = list(afwateringseenheden_laterals["code"].values)

    rr_input_zomer = rr_input_zomer[rr_input_zomer["ID_RR_KNOOP"].isin(list_rr_nodes_codes)]
    rr_input_winter = rr_input_winter[rr_input_winter["ID_RR_KNOOP"].isin(list_rr_nodes_codes)]
    
    rr_input_zomer.to_file(dir_scenario_input / "RR_input_ZOMER.gpkg", driver="GPKG")
    rr_input_winter.to_file(dir_scenario_input / "RR_input_WINTER.gpkg", driver="GPKG")
    
    gdfs_rr_input = generate_rr_unpaved_ernst_from_input(dir_scenario_input, dir_scenario_output)
    scenario_gdf_input[scenario] = gdfs_rr_input

Scenario: REF
- season: zomer
- season: winter
Scenario: SCEN
- season: zomer
- season: winter


In [18]:
gdfs_rr_input["zomer"]["unpaved"].iloc[0]

id                             AE54980042_OLF_nee_Drainw_laag
total_area                                             692500
lu_areas                 692500 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
surface_level                                        11.14082
soiltype                                                  114
surface_storage                                            10
infiltration_capacity                                      20
initial_gwd                                           2.35548
meteo_area                                      206725_447793
px                                                   205887.5
py                                                   447537.5
boundary_node              lat_AE54980042_OLF_nee_Drainw_laag
boundary_waterlevel                                   8.68534
geometry                            POINT (205887.5 447537.5)
Name: AE54980042_OLF_nee_Drainw_laag, dtype: object

#### PREPARE INPUT FOR PROJECT/PILOT AREAS

In [19]:
list_scenarios = list(scenario_gdf_input.keys())
seasons = scenario_gdf_input[list_scenarios[0]].keys()


for i, project_area in project_areas.iterrows():
    print(project_area.area_id)
    dir_input_area = dir_input / "rr_input_area_scenario" / f"gebied_{project_area.area_id}"
    if not dir_input_area.exists():
        dir_input_area.mkdir(parents=True, exist_ok=True)
    
    for scenario in list_scenarios:
        print(scenario)
        dir_input_area_scenario = dir_input_area / scenario
        if not dir_input_area_scenario.exists():
            dir_input_area_scenario.mkdir(parents=True, exist_ok=True)
        
        for season in seasons:
            print(season)
            # rr_unpaved
            scenario_gdf_unpaved_area = scenario_gdf_input[scenario][season]["unpaved"].clip(project_area.geometry)
            scenario_gdf_unpaved_area.to_file(
                dir_input_area_scenario / f"gdf_unpaved_{season}.gpkg", 
                layer=f"gdf_unpaved_{season}", 
                driver="GPKG"
            )

            # rr_unpaved
            scenario_df_unpaved_area = scenario_gdf_unpaved_area.drop(columns="geometry")
            scenario_df_unpaved_area.to_csv(dir_input_area_scenario / f"df_unpaved_{season}.csv")

            # rr_ernst
            sel_ernst = scenario_gdf_input[scenario][season]["ernst"].index.str.replace("ernst_", "unp_")
            scenario_df_ernst_area = scenario_gdf_input[scenario][season]["ernst"].loc[sel_ernst.isin(scenario_gdf_unpaved_area.index)]
            scenario_df_ernst_area.to_csv(dir_input_area_scenario / f"df_ernst_{season}.csv")

    list_rr_nodes_codes = list(scenario_gdf_unpaved_area.index.str.replace("unp_", ""))
    
    # afwateringseenheden
    afwateringseenheden_laterals = gpd.read_file(dir_scenario_input / "afwateringseenheden_laterals.gpkg")
    afwateringseenheden_laterals_area = afwateringseenheden_laterals[afwateringseenheden_laterals["code"].isin(list_rr_nodes_codes)]
    afwateringseenheden_laterals_area.to_file(dir_input_area / f"afwateringseenheden.gpkg", layer=f"afwateringseenheden", driver="GPKG")

    # watergang
    watergang_area = watergang.clip(project_area.geometry).explode()
    watergang_area.to_file(dir_input_area / f"watergang.gpkg", layer=f"watergang", driver="GPKG")

    # kwel/wegzijging (seepage)
    seepage_area = pd.DataFrame(
        columns=["sep_" + afw_eenheid for afw_eenheid in list_rr_nodes_codes],
        index=pd.date_range(start=start_date, end=end_date, freq="MS")
    ).fillna(0.0)
    seepage_area.to_csv(dir_input_area / f"seepage.csv")

0
REF
zomer
winter
SCEN
zomer
winter
1
REF
zomer
winter
SCEN
zomer
winter
2
REF
zomer
winter
SCEN
zomer
winter
3
REF
zomer
winter
SCEN
zomer
winter


#### METEO DATA: PRECIPITATION AND EVAPORATION (ALLES)

In [ ]:
# METEO - Neerslag
dir_neerslag_data = Path(dir_data_scenarios, "meteo", "neerslag_tijdreeksen\\output_tijdreeksen")

# select meteostations from scenario input
meteo_stations = scenario_gdf_input[list_scenarios[0]]["zomer"]["input"]["MeteoStationName"].unique()
neerslag_tijdseries = pd.DataFrame()

for meteo_station in meteo_stations:
    path_tijdserie = Path(dir_neerslag_data, meteo_station + ".txt")
    if path_tijdserie.exists():
        tijdserie = pd.read_csv(path_tijdserie, sep=";", index_col=0, parse_dates=["YYYYMMDDHH"], date_format="%Y%m%d%H")
        neerslag_tijdseries[meteo_station] = tijdserie

neerslag_tijdseries = neerslag_tijdseries.loc[start_date:end_date]

write_bui(
    neerslag_tijdseries, 
    Path(dir_input_scenarios, "meteo", "METEO_NEERSLAG.BUI"), 
    timestep_seconds=3600
)

# neerslag_tijdseries

In [ ]:
# METEO - Verdamping
dir_meteo = Path(dir_data_scenarios, "meteo")
file_evp = "verdamping_hupsel.xlsx"

evp = pd.read_excel(dir_meteo / file_evp, index_col=0, parse_dates=True)
evp.columns = ["evp"]

evp["jaar"] = evp.index.year
evp["maand"] = evp.index.month
evp["dag"] = evp.index.day
evp = evp[["jaar", "maand", "dag", "evp"]]

evp = evp.loc[start_date:end_date]

header_evp_file = (
    "*evpsfile Verdamping Hupsel\n"
    "*Meteo data: evaporation intensity in mm/day\n"
    "*First record: start date, data in mm/day\n"
    "*Datum (year month day), evp (mm/dag) voor elk weerstation\n"
    "*jaar maand dag evp[mm]\n"
)

output_file = Path(dir_input_scenarios, "meteo", "METEO_verdamping.EVP")

with open(output_file, "w") as f:
    f.write(header_evp_file)
    evp.to_string(
        f,
        index=False,
        header=False,
        formatters={"evp": "{:.3f}".format}
    )

# evp

#### METEO: NEERSLAG EN VERDAMPING (SPECIFIEK PER GEBIED)

In [ ]:
list_scenarios = list(scenario_gdf_input.keys())
seasons = scenario_gdf_input[list_scenarios[0]].keys()

Neerslag

In [ ]:
# METEO - Neerslag
dir_neerslag_data = Path(dir_data_scenarios, "meteo", "neerslag_tijdreeksen\\output_tijdreeksen")

# select meteostations from scenario input
meteo_stations = scenario_gdf_input[list_scenarios[0]]["zomer"]["input"]["MeteoStationName"].unique()
neerslag_tijdseries = pd.DataFrame()

for i, meteo_station in enumerate(meteo_stations):
    print(i, len(meteo_stations))
    path_tijdserie = Path(dir_neerslag_data, meteo_station + ".txt")
    if path_tijdserie.exists():
        tijdserie = pd.read_csv(path_tijdserie, sep=";", index_col=0, parse_dates=["YYYYMMDDHH"], date_format="%Y%m%d%H")
        neerslag_tijdseries[meteo_station] = tijdserie

neerslag_tijdseries = neerslag_tijdseries.loc[start_date:end_date]

for i, area in project_areas.iterrows():
    print(i, area.area_id)
    dir_input_area = dir_input / "rr_input_area_scenario" / f"gebied_{area.area_id}"
    dir_input_area_meteo = dir_input_area / "meteo"
    if not dir_input_area_meteo.exists():
        dir_input_area_meteo.mkdir(parents=True, exist_ok=True)

    for scenario in list_scenarios:
        for season in seasons:
            # meteo
            scenario_gdf_input_area = scenario_gdf_input[scenario][season]["input"].clip(area.geometry)
            meteo_stations_area = list(scenario_gdf_input_area["MeteoStationName"].unique())
            precipitation_timeseries_area = neerslag_tijdseries[meteo_stations_area]
            write_bui(
                precipitation_timeseries_area, 
                dir_input_area_meteo / "METEO_NEERSLAG.BUI", 
                timestep_seconds=3600
            )
            break
        break

VERDAMPING

In [ ]:
# # METEO - Verdamping
dir_meteo = Path(dir_data_scenarios, "meteo")
file_evp = "verdamping_hupsel.xlsx"

evp = pd.read_excel(dir_meteo / file_evp, index_col=0, parse_dates=True)
evp.columns = ["evp"]

evp["jaar"] = evp.index.year
evp["maand"] = evp.index.month
evp["dag"] = evp.index.day
evp = evp[["jaar", "maand", "dag", "evp"]]

evp = evp.loc[start_date:end_date]

header_evp_file = (
    "*evpsfile Verdamping Hupsel\n"
    "*Meteo data: evaporation intensity in mm/day\n"
    "*First record: start date, data in mm/day\n"
    "*Datum (year month day), evp (mm/dag) voor elk weerstation\n"
    "*jaar maand dag evp[mm]\n"
)

for i, area in project_areas.iterrows():
    print(area.area_id)
    dir_input_area = dir_input / "rr_input_area_scenario" / f"gebied_{area.area_id}"
    dir_input_area_meteo = dir_input_area / "meteo"
    if not dir_input_area_meteo.exists():
        dir_input_area_meteo.mkdir(parents=True, exist_ok=True)

    evp_file = Path(dir_input_area_meteo, "METEO_VERDAMPING.EVP")

    with open(evp_file, "w") as f:
        f.write(header_evp_file)
        evp.to_string(
            f,
            index=False,
            header=False,
            formatters={"evp": "{:.3f}".format}
        )